In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from utils.TUEP import TUHEpilepsy
from utils.photostim import extract_photostim_chunks
import mne
from copy import deepcopy, copy

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
np.set_printoptions(threshold=1000, edgeitems=20, linewidth=1000)

TUEP_PATH='/space/gzanardini/tuh_eeg/tuh_eeg_epilepsy'

tuep=TUHEpilepsy(path=TUEP_PATH,set_montage=False,rename_channels=True,target_name='epilepsy', n_jobs=1, preload=False)

In [3]:
def select_by_duration(dataset,tmin=10):
    new_dataset=deepcopy(dataset)
    new_dataset.datasets=[sample for sample in dataset.datasets if sample.raw.n_times/sample.raw.info['sfreq'] >= tmin]
    return new_dataset

def select_by_channel(dataset,channels):
    new_dataset=deepcopy(dataset)
    new_dataset.datasets=[sample for sample in dataset.datasets if set(channels).issubset(ch_name.upper() for ch_name in sample.raw.ch_names)]
    return new_dataset

channels = ['FP1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'FZ', 'CZ',
            'PZ', 'FP2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2', 'PHOTIC PH']

def preprocess(in_dataset, target_freq=None, photic_ph=True):
    dataset=deepcopy(in_dataset)
    if photic_ph:
        channels = ['FP1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'FZ', 'CZ',
            'PZ', 'FP2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2', 'PHOTIC PH']
    else:
        channels = ['FP1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'FZ', 'CZ',
            'PZ', 'FP2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2']
        
    for i, sample in enumerate(dataset.datasets):
        if set(channels[:-1]).issubset(ch_name.upper() for ch_name in sample.raw.ch_names) == False:
            print(f"Channels missing in {i}")
            print(set(channels) - set(sample.raw.ch_names))
            continue

        sample.raw.rename_channels(lambda x: x.upper(),verbose=False)
        
        #check for PHOTIC PH channel
        if 'PHOTIC PH' in sample.raw.ch_names and photic_ph == True:
            #set the channel type to stim
            sample.raw.set_channel_types({'PHOTIC PH': 'stim'}, on_unit_change="ignore")
        elif 'PHOTIC PH' not in sample.raw.ch_names and photic_ph == True: 
            sample.raw.load_data()
            sample.raw.add_channels([mne.io.RawArray(np.zeros((1, sample.raw.n_times)), mne.create_info(['PHOTIC PH'], sample.raw.info['sfreq'], ['stim']), verbose=False)],force_update_info=True)
            print(f"Added PHOTIC PH channel to {i}")

        sample.raw.pick(channels)
        sample.raw.reorder_channels(channels)
        sample.raw.load_data()
        sample.raw.filter(0.1,(sample.raw.info['sfreq']/2)-0.1,verbose=False, n_jobs=8, l_trans_bandwidth=0.1)
        #sample.raw.filter(0.1,(target_freq/2)-0.1,verbose=False, n_jobs=8, l_trans_bandwidth=0.1)
        sample.raw.notch_filter(60,verbose=False, n_jobs=8)

        if sample.raw.info['sfreq'] != target_freq and target_freq is not None:
            sample.raw.resample(target_freq, n_jobs=8)
            print(f"Resampled {i} to {target_freq} Hz")

    return dataset

tuep_test=select_by_channel(tuep,channels)

tuep_test=preprocess(tuep_test, target_freq=250, photic_ph=True)

tuep_test=select_by_duration(tuep_test, tmin=10)

# savedir='/space/gzanardini/tuh_eeg/preprocessed/fullv2/'
# tuep_test.save(savedir, overwrite=True)

Reading 0 ... 303499  =      0.000 ...  1213.996 secs...
Reading 0 ... 332249  =      0.000 ...  1328.996 secs...
Reading 0 ... 376499  =      0.000 ...  1505.996 secs...
Reading 0 ... 298499  =      0.000 ...  1193.996 secs...
Reading 0 ... 3499  =      0.000 ...    13.996 secs...
Reading 0 ... 277999  =      0.000 ...  1111.996 secs...
Reading 0 ... 97249  =      0.000 ...   388.996 secs...
Reading 0 ... 368499  =      0.000 ...  1473.996 secs...
Reading 0 ... 306999  =      0.000 ...  1227.996 secs...
Reading 0 ... 320499  =      0.000 ...  1281.996 secs...
Reading 0 ... 318749  =      0.000 ...  1274.996 secs...
Reading 0 ... 379499  =      0.000 ...  1517.996 secs...
Reading 0 ... 21999  =      0.000 ...    87.996 secs...
Reading 0 ... 18999  =      0.000 ...    75.996 secs...
Reading 0 ... 46249  =      0.000 ...   184.996 secs...
Reading 0 ... 46249  =      0.000 ...   184.996 secs...
Reading 0 ... 47249  =      0.000 ...   188.996 secs...
Reading 0 ... 31249  =      0.000 ...  

In [4]:
def remove_calibration_signal(eeg, amplitude, threshold=1):
    for t in range(eeg.shape[1]):
        if np.abs(eeg[19, t]) < amplitude - threshold:
            start = t
            break
        else:
            start = 0
    return eeg[:, start:]

In [ ]:
samples_to_discard = {
    "aaaaabju": {"session": 1, "segment": 0},
    "aaaaaflb": {"session": 1, "segment": 0},
    "aaaaagxr": {"session": 1, "segment": 1},
    "aaaaajgn": {"session": 1, "segment": 0},
    "aaaaajrh": {"session": 2, "segment": 0}
} ### Manually identified samples, despite the presence of the PHOTIC PH channel, do not contain a clear photic stimulation signal.

In [ ]:
def preproc_photostim(dataset, fs_target=250):

    channels = ['Fp1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'Fz', 'Cz',
                'Pz', 'Fp2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2', 'PHOTIC PH']

    channels_dict = {ch: 'eeg' for ch in channels[:-1]}
    channels_dict['PHOTIC PH'] = 'stim'

    new_df=pd.DataFrame(index=range(100), columns=dataset.description.columns) 
    samples_with_stim=[]
    ctr=0
    for i, recording in enumerate(dataset.datasets):

    #discard samples which perfectly match the samples_to_discard
        if recording.description['subject'] in samples_to_discard.keys():
            if recording.description['session'] == samples_to_discard[recording.description['subject']]['session']:        
                if recording.description['segment'] == samples_to_discard[recording.description['subject']]['segment']:
                    print(f"##### Discarding {recording.description['subject']} - {recording.description['session']} - {recording.description['segment']}")
                    continue

        if 'PHOTIC PH' in recording.raw.ch_names:
            recording.raw.pick(channels)
            recording.raw.reorder_channels(channels)
            recording.raw.set_channel_types(channels_dict, on_unit_change="warn")

            temp=remove_calibration_signal(recording.raw.get_data(), 100)
            temp[19]=temp[19]*1e-6

            if not np.all(temp[19] == 0) and temp.shape[1]*1/recording.raw.info['sfreq']>=30:

                #make a new raw object with the new data
                recording.raw=mne.io.RawArray(temp, recording.raw.info, verbose=False)
                #if not np.all(recording.raw['PHOTIC PH'][0][0] == 0) and recording.raw.last_samp*1/recording.raw.info['sfreq']>=30:
                #print(f'Length of recording {i} : {recording.raw.last_samp*1/recording.raw.info["sfreq"]}')
                
                recording.raw.notch_filter(60,verbose=False, n_jobs=8)

                if recording.raw.info['sfreq'] != fs_target and fs_target is not None:
                    recording.raw.resample(fs_target, n_jobs=8)
                    print(f"Resampled {i} to {fs_target} Hz")

                samples_with_stim.append(recording.raw)
                new_df.loc[ctr]=dataset.description.loc[i]
                ctr+=1

    new_df=new_df.dropna()

    return new_df, samples_with_stim

stim_df, stim_samples=preproc_photostim(tuep, fs_target=250)

print(len(stim_df), len(stim_samples))

In [ ]:
import pickle as pkl
with open('photostimulation/temp_data/stim_samples.pkl', 'wb') as f:
    pkl.dump(stim_samples, f)

stim_df.to_csv('photostimulation/temp_data/stim_df.csv', index=False)

In [ ]:
photostim_periods = []
no_photostim_periods=[]
intervals=[]

wins_df=pd.DataFrame(index=range(150), columns=stim_df.columns)
no_ph_df=pd.DataFrame(index=range(150), columns=stim_df.columns)

for i, edf in enumerate(stim_samples):

    sample=edf.get_data()
    print(f'Length of the sample {i}: {sample.shape[1]/250}')
    print(f'Channel order: {edf.ch_names}')

    for t in range(sample.shape[1]): # find the first non-zero value
        if np.abs(sample[19, t]) > 0:
            start = t
            break
    for t in range(sample.shape[1]-1, 0, -1): # find the last non-zero value
        if np.abs(sample[19, t]) > 0:
            end = t
            break

    start-=250 # add 1s before the first non-zero value
    end+=250 # add 1s after the last non-zero value

    temp=sample[:, start:end] # get the segment

    print(f'Start: {start}, End: {end}, Length: {temp.shape[1]/250}')
    intervals.append((start, end))
    '''
    print('Length of the segment - sample :',i,' ', (end-start)/250)
    plt.figure(figsize=(10, 2))
    plt.title(f'Subject: {photostim_df.loc[i]['subject']} - Session: {photostim_df.loc[i]['session']} - Segment {photostim_df.loc[i]['segment']} - Epilepsy: {photostim_df.loc[i]['epilepsy']}')
    plt.plot(temp[19,:])
    plt.show()'''

    photostim_periods.append(temp)
    wins_df.loc[i]=stim_df.loc[i]

    del temp

    temp1=sample[:, :start]
    temp2=sample[:, end:]
    temp=np.concatenate((temp1, temp2), axis=1)

    no_photostim_periods.append(temp)
    no_ph_df.loc[i]=stim_df.loc[i]

    del temp1, temp2, temp

#duplicate .iloc[10] since the recording has 2 different photostimulation periods
temp_df=wins_df.copy().iloc[:11]
temp_df2=wins_df.copy().iloc[10:]

wins_df=pd.concat([temp_df, temp_df2], ignore_index=True, axis=0)
wins_df.dropna(inplace=True)

print('Length of the photostimulation periods: ',len(photostim_periods))
print('Length of the windows descriptions: ',len(wins_df))

long_sample=copy(photostim_periods[10])

for t in range(long_sample.shape[1]):
    if np.abs(long_sample[19, t]) > 0:
        start = t
        break
for t in range(long_sample.shape[1]-1, 0, -1):
    end=long_sample.shape[1]
    if np.abs(long_sample[19, t]) > 0:
        end = t
        break

midpoint = (start+end)//2
first_part = copy(long_sample[:, start:midpoint+250])
second_part = copy(long_sample[:, midpoint-250:end+250])

print(f'FULL SAMPLE - Start: {start}, End: {end}, Midpoint: {midpoint}')

plt.figure(figsize=(20, 2))
plt.plot(long_sample[19,:])
plt.title('Long sample')
plt.show()

for t in range(first_part.shape[1]):
    if np.abs(first_part[19, t]) > 0:
        start = t
        break
for t in range(first_part.shape[1]-1, 0, -1):
    if np.abs(first_part[19, t]) > 0:
        end = t
        break

print(f'1st part: Start: {start}, End: {end}')

end1=end+250

first_part = copy(first_part[:, start:end1])

print('First part:', first_part.shape[1])
print(f'Sanity check length: {(end1-start)}')

for t in range(second_part.shape[1]):
    if np.abs(second_part[19, t]) > 0:
        start = t
        break
for t in range(second_part.shape[1]-1, 0, -1):
    if np.abs(second_part[19, t]) > 0:
        end = t
        break

second_part = copy(second_part[:, start-250:end+250])

plt.figure(figsize=(20, 4))
plt.subplot(2, 1, 1)
plt.plot(first_part[19,:])
plt.title('First part')
plt.subplot(2, 1, 2)
plt.plot(second_part[19,:])
plt.title('Second part')
plt.show()

photostim_periods[10]=first_part
photostim_periods.insert(11, second_part)

print('SANITY CHECK')
print('Length of the photostimulation periods: ',len(photostim_periods))
print('Length of the windows descriptions: ',len(wins_df))

In [ ]:
import pickle
import os

no_ph_df.dropna(inplace=True)

print('Number of samples with photic stimulation:', len(no_photostim_periods))
print('Length of the photostimulation descriptions: ',len(no_ph_df))

# Create the directory if it does not exist
# Save no_photostim_periods
with open('photostimulation/data/no_photostim_periods.pkl', 'wb') as f:
    pickle.dump(no_photostim_periods, f)

no_ph_df.to_csv('photostimulation/data/no_photostim.csv')

In [ ]:
# photostim_chunks, chunks_df = extract_photostim_chunks(
#     photostim_periods,
#     wins_df,
#     chunk_length=19.3,
#     fs=250,
#     start_sample=100,
#     skip_indices=(35,),
#     anchor_channel=19,
#     trim_samples=2550,
#     channel_offset=25,
# )

# print('Length of the photostimulation chunks: ',len(photostim_chunks))
# print('Length of the chunks descriptions: ',len(chunks_df))